# Sistema para apuração de fiscal - Empresa vs Contabilidade

## Bibliotecas

In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# Frontend

In [3]:
def selecionar_arquivo(entry_widget):
    """Abre a janela para escolher o arquivo Excel correspondente"""
    arquivo = filedialog.askopenfilename(
        title="Selecione a planilha",
        filetypes=[("Arquivos Excel", "*.xlsx *.xls")]
    )
    if arquivo:
        entry_widget.delete(0, tk.END)
        entry_widget.insert(0, arquivo)

def executar_auditoria():
    caminho_empresa = entry_empresa.get()
    caminho_icms = entry_icms.get()
    caminho_ipi = entry_ipi.get()
    nome_saida = entry_saida.get().strip()

    # Validações iniciais
    if not caminho_empresa or not caminho_icms or not caminho_ipi:
        messagebox.showwarning("Atenção", "Por favor, selecione os três arquivos de entrada!")
        return
    
    if not nome_saida:
        nome_saida = "apuracao.xlsx"
    elif not nome_saida.endswith(".xlsx"):
        nome_saida += ".xlsx"

    try:
        # 1. Leitura das planilhas
        df_empresa = pd.read_excel(caminho_empresa)
        df_contabilidade_icms = pd.read_excel(caminho_icms)
        df_contabilidade_ipi = pd.read_excel(caminho_ipi)

        # 2. Renomeia as colunas de valor para evitar conflitos de nomes
        df_empresa = df_empresa.rename(columns={
            'Valor Contábil': 'vl_cont_empresa',
            'Valor Icms': 'vl_icms_empresa',
            'Valor IPI': 'vl_ipi_empresa'
            'Número': 'nf_empresa'
        })
        df_contabilidade_icms = df_contabilidade_icms.rename(columns={
            'Vlr Contábil Kato ICMS': 'vl_cont_contabilidade_icms',
            'Valor ICMS Kato': 'vl_icms_contabilidade',
            'Número': 'nf_icms_contabilidade'
        })
        df_contabilidade_ipi = df_contabilidade_ipi.rename(columns={
            'vl_cont_ipi_kato': 'vl_cont_contabilidade_ipi',
            'vl_ipi_kato': 'vl_ipi_contabilidade',
            'Número': 'nf_ipi_contabilidade'
        })

        # 3. Padronização e criação do id_unico
        df_empresa['id_unico'] = df_empresa['nf_empresa'].astype(str) + '|' + df_empresa['Natureza'].astype(str)
        df_contabilidade_icms['id_unico'] = df_contabilidade_icms['nf_icms_contabilidade'].astype(str) + '|' + df_contabilidade_icms['Natureza'].astype(str)
        df_contabilidade_ipi['id_unico'] = df_contabilidade_ipi['nf_ipi_contabilidade'].astype(str) + '|' + df_contabilidade_ipi['Natureza'].astype(str)


        # 4. Removendo colunas desnecessárias
        # 4.1 Empresa
        delete_columns_empresa = ['Espécie', 
                          'Série', 
                          'Emissão', 
                          'Entrada', 
                          'Cód Emitente', 
                          'UF', 
                          'Incidência',
                          '%',
                          'Base Cálc.',
                          'Isento',
                          'Outros',
                          'Incidência.1',
                          '%.1',
                          'Base Cálc..1',
                          'Observação Substituição Tributária',
                          'Isento.1',
                          'Outros.1',
                          'Observação']
        df_empresa = df_empresa.drop(columns=delete_columns_empresa)

        # 4.2 Contabilidae - ICMS
        df_contabilidade_icms = df_contabilidade_icms.drop(columns=['Nr Lcto', 
                                          'Pessoa', 
                                          'Data Lcto', 
                                          'Esp', 
                                          'Série',
                                          'Base ICMS', 
                                          'Isentas ICMS', 
                                          'Outras ICMS'])

        # 4.3 Contabilidade - IPI
        df_contabilidade_ipi = df_contabilidade_ipi.drop(columns=['Nr Lcto', 
                                        'Fornecedor', 
                                        'Data Lcto', 
                                        'Esp', 
                                        'Série',
                                        'Base IPI', 
                                        'Isentas IPI', 
                                        'Outras IPI'])

        # 5.0 Removendo linhas em branco
        df_empresa = df_empresa.dropna()
        df_contabilidade_icms = df_contabilidade_icms.dropna()
        df_contabilidade_ipi = df_contabilidade_ipi.dropna()

        # 6.0 Preechendo valores em branco com 0
        df_empresa = df_empresa.fillna(0)
        df_contabilidade_icms = df_contabilidade_icms.fillna(0)
        df_contabilidade_ipi = df_contabilidade_ipi.fillna(0)

        # 7.0 Limpando as tabelas da contabilidade
        df_contabilidade_icms = df_contabilidade_icms[df_contabilidade_icms['cfop_icms_contabilidade'] != 'Natureza']
        df_contabilidade_ipi = df_contabilidade_ipi[df_contabilidade_ipi['cfop_ipi_contabilidade'] != 'Natureza']

        # 8.0 Formatação dos dados
        # 8.1 Tabela ICMS
        colunas_icms_float = ['vl_icms_contabilidade', 'vl_cont_icms_contabilidade']
        colunas_icms_int = ['cfop_icms_contabilidade','nf_icms_contabilidade']
        for col in colunas_icms_int:
            df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('int')

        for col in colunas_icms_float:
            df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('float')

        # 8.2 Tabela IPI
        colunas_ipi_float = ['vl_ipi_contabilidade', 'vl_cont_ipi_contabilidade']
        colunas_ipi_int = ['cfop_ipi_contabilidade', 'nf_ipi_contabilidade']
        for col in colunas_ipi_int:
            df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('int')

        for col in colunas_ipi_float:
            df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('float')

        # 4. Agrupamento para somar notas iguais na mesma natureza (evitando duplicações)
        df_empresa_g = df_empresa.groupby('id_unico', as_index=False).sum(numeric_only=True)
        df_icms_g = df_contabilidade_icms.groupby('id_unico', as_index=False).sum(numeric_only=True)
        df_ipi_g = df_contabilidade_ipi.groupby('id_unico', as_index=False).sum(numeric_only=True)

        # 5. Criação da Lista Mestre
        lista_mestre = pd.concat([
            df_empresa_g['id_unico'],
            df_icms_g['id_unico'],
            df_ipi_g['id_unico']
        ]).dropna().drop_duplicates()

        df_mestre = pd.DataFrame({'id_unico': lista_mestre})

        # 6. Merge (Left Join)
        df_mestre = pd.merge(df_mestre, df_empresa_g, on='id_unico', how='left')
        df_mestre = pd.merge(df_mestre, df_icms_g, on='id_unico', how='left')
        df_mestre = pd.merge(df_mestre, df_ipi_g, on='id_unico', how='left')

        # 7. Limpeza imediata dos vazios (NaN) com 0 para evitar erros nas diferenças
        df_mestre = df_mestre.fillna(0)

        # 8. Divisão do ID em 'nf' e 'natureza' e conversão para inteiros limpos
        df_mestre[['nf', 'natureza']] = df_mestre['id_unico'].str.split('|', expand=True)
        df_mestre['nf'] = pd.to_numeric(df_mestre['nf'], errors='coerce').astype('Int64')
        df_mestre['natureza'] = pd.to_numeric(df_mestre['natureza'], errors='coerce').astype('Int64')

        # 9. Cálculo das Diferenças (Empresa vs Contabilidade correspondente)
        df_mestre['Dif Valor Contabil'] = df_mestre['vl_cont_empresa'] - df_mestre['vl_cont_contabilidade_icms']
        df_mestre['Dif Valor ICMS'] = df_mestre['vl_icms_empresa'] - df_mestre['vl_icms_contabilidade']
        df_mestre['Dif Valor IPI'] = df_mestre['vl_ipi_empresa'] - df_mestre['vl_ipi_contabilidade']

        # Arredonda as diferenças para 2 casas decimais (eliminando dízimas E-12)
        cols_dif = ['Dif Valor Contabil', 'Dif Valor ICMS', 'Dif Valor IPI']
        for col in cols_dif:
            df_mestre[col] = df_mestre[col].round(2)

        # 10. Classificação (OK ou Valor Divergente)
        df_mestre['Classificacao'] = 'Valor Divergente'
        df_mestre.loc[df_mestre[cols_dif].eq(0).all(axis=1), 'Classificacao'] = 'OK'

        # 11. Organização das colunas principais no início
        cols_principais = ['id_unico', 'nf', 'natureza', 'Classificacao']
        outras_cols = [c for c in df_mestre.columns if c not in cols_principais]
        df_mestre = df_mestre[cols_principais + outras_cols]

        # 12. Criação da Aba de Resumo Executivo (Indicadores nas colunas, valores nas linhas)
        qtde_nf_empresa = (df_mestre['vl_cont_empresa'] != 0).sum()
        qtde_nf_icms_kato = (df_mestre['vl_cont_contabilidade_icms'] != 0).sum()

        valor_contabil_empresa = df_mestre['vl_cont_empresa'].sum()
        valor_vl_cont_contabilidade_icms = df_mestre['vl_cont_contabilidade_icms'].sum()
        
        valor_icms_empresa = df_mestre['vl_icms_empresa'].sum()
        valor_icms_kato = df_mestre['vl_icms_contabilidade'].sum()

        valor_ipi_empresa = df_mestre['vl_ipi_empresa'].sum()
        valor_ipi_kato = df_mestre['vl_ipi_contabilidade'].sum()

        df_resumo = pd.DataFrame({
            'Qtde Nfs Empresa': [qtde_nf_empresa],
            'Qtde Nfs Contabilidade': [qtde_nf_icms_kato],
            'Diferença qtde Nfs': [qtde_nf_empresa - qtde_nf_icms_kato],
            'Valor Total Contabil Empresa': [valor_contabil_empresa],
            'Valor Total Contabil Contabilidade': [valor_vl_cont_contabilidade_icms],
            'Diferença valor total Contabil': [valor_contabil_empresa - valor_vl_cont_contabilidade_icms],
            'Valor Total ICMS Empresa': [valor_icms_empresa],
            'Valor Total ICMS Contabilidade': [valor_icms_kato],
            'Diferença Valor total ICMS': [valor_icms_empresa - valor_icms_kato],
            'Valor Total IPI Empresa': [valor_ipi_empresa],
            'Valor Total IPI Contabilidade': [valor_ipi_kato],
            'Diferença Valor total IPI': [valor_ipi_empresa - valor_ipi_kato]
        })

        # 13. Pergunta ao usuário onde deseja salvar o arquivo final
        caminho_salvar = filedialog.asksaveasfilename(
            defaultextension=".xlsx",
            initialfile=nome_saida,
            filetypes=[("Arquivo Excel", "*.xlsx")]
        )

        if caminho_salvar:
            # Salva o Excel com as duas abas
            with pd.ExcelWriter(caminho_salvar, engine='openpyxl') as writer:
                df_mestre.to_excel(writer, sheet_name='Auditoria_Detalhada', index=False)
                df_resumo.to_excel(writer, sheet_name='Resumo_Executivo', index=False)

            # Reabre para aplicar as cores verde/vermelho na aba detalhada
            wb = load_workbook(caminho_salvar)
            ws = wb['Auditoria_Detalhada']

            fill_verde = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
            fill_vermelho = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")

            # Descobre a coluna 'Classificacao'
            col_idx_status = None
            for col_i, cell in enumerate(ws[1], 1):
                if cell.value == 'Classificacao':
                    col_idx_status = col_i
                    break

            if col_idx_status:
                for row in range(2, ws.max_row + 1):
                    cell = ws.cell(row=row, column=col_idx_status)
                    if cell.value == 'OK':
                        cell.fill = fill_verde
                    elif cell.value == 'Valor Divergente':
                        cell.fill = fill_vermelho

            wb.save(caminho_salvar)
            messagebox.showinfo("Sucesso!", f"Auditoria concluída e salva com sucesso em:\n{caminho_salvar}")

    except Exception as e:
        messagebox.showerror("Erro no Processamento", f"Ocorreu um erro durante a execução:\n{str(e)}")

# --- CONSTRUÇÃO DA TELA (GUI) ---
root = tk.Tk()
root.title("Ferramenta de Auditoria Fiscal - Contabilidade")
root.geometry("600x420")
root.resizable(False, False)

tk.Label(root, text="Automação de Cruzamento Fiscal", font=("Arial", 14, "bold")).pack(pady=15)

# Campo 1: Injetaq
tk.Label(root, text="Livro de Entradas (Injetaq / Empresa):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25)
f1 = tk.Frame(root)
f1.pack(fill="x", padx=25, pady=3)
entry_empresa = tk.Entry(f1, width=60)
entry_empresa.pack(side="left", padx=(0, 5))
tk.Button(f1, text="Procurar", command=lambda: selecionar_arquivo(entry_empresa)).pack(side="left")

# Campo 2: ICMS
tk.Label(root, text="Livro de ICMS (Contabilidade):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
f2 = tk.Frame(root)
f2.pack(fill="x", padx=25, pady=3)
entry_icms = tk.Entry(f2, width=60)
entry_icms.pack(side="left", padx=(0, 5))
tk.Button(f2, text="Procurar", command=lambda: selecionar_arquivo(entry_icms)).pack(side="left")

# Campo 3: IPI
tk.Label(root, text="Livro de IPI (Contabilidade):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
f3 = tk.Frame(root)
f3.pack(fill="x", padx=25, pady=3)
entry_ipi = tk.Entry(f3, width=60)
entry_ipi.pack(side="left", padx=(0, 5))
tk.Button(f3, text="Procurar", command=lambda: selecionar_arquivo(entry_ipi)).pack(side="left")

# Campo 4: Nome do arquivo final
tk.Label(root, text="Nome do Relatório de Saída (.xlsx):", anchor="w", font=("Arial", 9, "bold")).pack(fill="x", padx=25, pady=(8,0))
entry_saida = tk.Entry(root, width=67)
entry_saida.pack(padx=25, pady=3, anchor="w")
entry_saida.insert(0, "auditoria_fiscal_resultado.xlsx")

# Botão Executar Principal
btn_executar = tk.Button(
    root, 
    text="Executar Auditoria e Gerar Excel", 
    bg="#2e7d32", 
    fg="white", 
    font=("Arial", 11, "bold"), 
    padx=10, 
    pady=5,
    command=executar_auditoria
)
btn_executar.pack(pady=20)

root.mainloop()



SyntaxError: invalid syntax (271570797.py, line 38)

## 1.0 Dados Empresa

### 1.1 Coleta de dados

In [ ]:
df_empresa = pd.read_excel("./dataset/Livro de Registros - Entrada.xlsx", header=1)

### 1.2 Alterar o nome das colunas

In [ ]:
df_empresa = df_empresa.rename(columns={
    'Número': 'nf_empresa',
    'Tributado': 'vl_icms_empresa',
    'Tributado.1': 'vl_ipi_empresa',
    'Nat. Op.': 'cfop_empresa',
    'Valor Contábil': 'vl_cont_empresa'
})

### 1.3 Remoção, limpeza e formatação dos dados

#### 1.3.1 Remover colunas desnecessárias

In [ ]:
delete_columns_empresa = ['Espécie', 
                          'Série', 
                          'Emissão', 
                          'Entrada', 
                          'Cód Emitente', 
                          'UF', 
                          'Incidência',
                          '%',
                          'Base Cálc.',
                          'Isento',
                          'Outros',
                          'Incidência.1',
                          '%.1',
                          'Base Cálc..1',
                          'Observação Substituição Tributária',
                          'Isento.1',
                          'Outros.1',
                          'Observação']
df_empresa = df_empresa.drop(columns=delete_columns_empresa)

#### 1.3.2 Remover linhas em branco

In [ ]:
df_empresa = df_empresa.dropna()

#### 1.3.3 Formatação dos dados

In [ ]:
df_empresa['nf_empresa'] = df_empresa['nf_empresa'].astype('int')
df_empresa['cfop_empresa'] = df_empresa['cfop_empresa'].astype('int')

#### 1.3.4 Criando a coluna ID unico

In [ ]:
df_empresa['id_unico_empresa'] = df_empresa['nf_empresa'].astype('str') + '|' + df_empresa['cfop_empresa'].astype('str')

#### 1.3.5 Somando os valores cuso os ID's unicos sejam iguais

In [ ]:
df_empresa = df_empresa.groupby('id_unico_empresa', as_index=False).sum(numeric_only=True)

#### 1.3.6 Preenchendo os valores em branco com 0

In [ ]:
df_empresa = df_empresa.fillna(0)

#### 1.3.7 Gerando os totais dos dados da empresa

In [ ]:
qtde_nf_empresa = (df_empresa['nf_empresa']).dropna().drop_duplicates().count()
valor_contabil_total_empresa = df_empresa['vl_cont_empresa'].sum()
valor_icms_total_empresa = df_empresa['vl_icms_empresa'].sum()
valor_ipi_total_empresa = df_empresa['vl_ipi_empresa'].sum()

## 2.0 Dados contabilidade (ICMS)

### 2.1 Coleta de dados

In [ ]:
df_contabilidade_icms = pd.read_excel("./dataset/analise_kato_icms.xlsx", header=6)


### 2.2 Alteraro nome das colunas

In [ ]:
df_contabilidade_icms = df_contabilidade_icms.rename(columns={
    'Número': 'nf_icms_contabilidade',
    'Natureza': 'cfop_icms_contabilidade',
    "Valor ICMS": 'vl_icms_contabilidade',
    'Vlr Contábil': 'vl_cont_icms_contabilidade'
})

### 2.3 Remoção, limpeza e formatação dos dados

#### 2.3.1 Removendo colunas desnecessárias

In [ ]:
df_contabilidade_icms = df_contabilidade_icms.drop(columns=['Nr Lcto', 
                                          'Pessoa', 
                                          'Data Lcto', 
                                          'Esp', 
                                          'Série',
                                          'Base ICMS', 
                                          'Isentas ICMS', 
                                          'Outras ICMS'])

#### 2.3.2 Removendo as linhas em branco

In [ ]:
df_contabilidade_icms = df_contabilidade_icms.dropna()

#### 2.3.3 Limpeza dos dados onde apareça a palavra 'Natureza'

In [ ]:
df_contabilidade_icms = df_contabilidade_icms[df_contabilidade_icms['cfop_icms_contabilidade'] != 'Natureza']

#### 2.3.4 Formatação dos dados

In [ ]:

colunas_icms_float = ['vl_icms_contabilidade', 'vl_cont_icms_contabilidade']
colunas_icms_int = ['cfop_icms_contabilidade','nf_icms_contabilidade']
for col in colunas_icms_int:
    df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('int')

for col in colunas_icms_float:
    df_contabilidade_icms[col] = df_contabilidade_icms[col].astype('float')

#### 2.3.5 Separando apenas os CFOP's menores ou iguais a 4000

In [ ]:
filtro = df_contabilidade_icms.query('cfop_icms_contabilidade <= 4000')
df_contabilidade_icms = filtro


#### 2.3.6 Criando a coluna ID unico

In [ ]:
df_contabilidade_icms['id_unico_ICMS'] = df_contabilidade_icms['nf_icms_contabilidade'].astype('str') + "|" + df_contabilidade_icms['cfop_icms_contabilidade'].astype('str')

#### 2.3.7 Gerando os totais dos dados da contabilidade ICMS

In [ ]:
qtde_nf_icms_contabilidade = (df_contabilidade_icms['nf_icms_contabilidade']).dropna().drop_duplicates().count()
valor_contabil_total_icms_contabilidade = df_contabilidade_icms['vl_cont_icms_contabilidade'].sum()
valor_icms_total_contabilidade = df_contabilidade_icms['vl_icms_contabilidade'].sum()
valor_icms_total_contabilidade


np.float64(92499.41)

## 3.0 Dados contabilidade (IPI)

### 3.1 Coleta de dados

In [ ]:
df_contabilidade_ipi = pd.read_excel("./dataset/analise_kato_ipi.xlsx", header=6)

### 3.2 Alterar o nome das colunas

In [ ]:
df_contabilidade_ipi = df_contabilidade_ipi.rename(columns={
    'Número': 'nf_ipi_contabilidade',
    'Natureza': 'cfop_ipi_contabilidade',
    'Valor IPI': 'vl_ipi_contabilidade',
    'Vlr Contabil': 'vl_cont_ipi_contabilidade'
})

### 3.3 Remoção, limpeza e formatação dos dados

#### 3.3.1 Removendo colunas desnecessárias

In [ ]:
df_contabilidade_ipi = df_contabilidade_ipi.drop(columns=['Nr Lcto', 
                                        'Fornecedor', 
                                        'Data Lcto', 
                                        'Esp', 
                                        'Série',
                                        'Base IPI', 
                                        'Isentas IPI', 
                                        'Outras IPI'])

#### 3.3.2 Removendo as linhas em branco

In [ ]:
df_contabilidade_ipi = df_contabilidade_ipi.dropna()

#### 3.3.3 Limpeza dos dados onde apareça a palavra 'Natureza'

In [ ]:
df_contabilidade_ipi = df_contabilidade_ipi[df_contabilidade_ipi['cfop_ipi_contabilidade'] != 'Natureza']

#### 3.3.4 Formatação dos dados

In [ ]:
colunas_ipi_float = ['vl_ipi_contabilidade', 'vl_cont_ipi_contabilidade']
colunas_ipi_int = ['cfop_ipi_contabilidade', 'nf_ipi_contabilidade']
for col in colunas_ipi_int:
    df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('int')

for col in colunas_ipi_float:
    df_contabilidade_ipi[col] = df_contabilidade_ipi[col].astype('float')

#### 3.3.5 Separando apenas os CFOP's menores ou iguais a 4000

In [ ]:
filtro = df_contabilidade_ipi.query('cfop_ipi_contabilidade <= 4000')
df_contabilidade_ipi = filtro
df_contabilidade_ipi['id_unico_IPI'] = df_contabilidade_ipi['nf_ipi_contabilidade'].astype('str') + "|" + df_contabilidade_ipi['cfop_ipi_contabilidade'].astype('str')

#### 3.3.7 Gerando os totais dos dados da contabilidade IPI

In [ ]:
valor_ipi_total_contabilidade = df_contabilidade_ipi['vl_ipi_contabilidade'].sum()

## 4.0 Criando o dataframe com a lista mestre com todas os ID's unicos dos 3 dataframes

### 4.1 Lista mestre - Faz a inclusão de todos os Id's unicos e deixa apenas aparecendo uma unica vez removendo os duplicados

In [ ]:
lista_mestre = pd.concat([
    df_empresa['id_unico_empresa'],
    df_contabilidade_icms['id_unico_ICMS'],
    df_contabilidade_ipi['id_unico_IPI']
]).dropna().drop_duplicates()

### 4.2 Faz a transformação da lista_mestre para um dataframe (df_mestre) para poder juntar com os outros dataframes

In [ ]:
df_mestre = pd.DataFrame({'id_unico': lista_mestre})

### 4.3 Remove as linhas em branco

In [ ]:
df_mestre = df_mestre.fillna(0)

### 4.4 Acrescenta ao dataframe df_mestre as colunas de nf e cfop

In [ ]:
df_mestre[['nf', 'cfop']] = df_mestre['id_unico'].str.split('|', expand=True)

## 5.0 União das Bases

### 5.1 Base Mestre + Base empresa

In [ ]:
df_mestre_e_empresa = pd.merge(
    df_mestre,          # Sua Tabela 1
    df_empresa,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_empresa',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.2 (Base Mestre + Base empresa) + Base ICMS contabilidade

In [ ]:
df_mestre_e_empresa_e_contabilidade_icms = pd.merge(
    df_mestre_e_empresa,          # Sua Tabela 1
    df_contabilidade_icms,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_ICMS',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.3 (Base Mestre + Base empresa + Base ICMS contabilidade + Base IPI contabilidade)

In [ ]:
df_unido = pd.merge(
    df_mestre_e_empresa_e_contabilidade_icms,          # Sua Tabela 1
    df_contabilidade_ipi,       # Sua Tabela 2 (substitua pelo nome real dela)
    left_on='id_unico', # Coluna de ID na Tabela 1
    right_on='id_unico_IPI',        # Coluna de ID na Tabela 2
    how='outer'                 # Garante que nada é descartado de nenhum dos lados
)

### 5.4 Transformação dos dados

In [ ]:
df_unido['cfop'] = pd.to_numeric(df_unido['cfop'], errors='coerce').astype('Int64')
df_unido['nf'] = pd.to_numeric(df_unido['nf'], errors='coerce').astype('Int64')
df_unido['nf_icms_contabilidade'] = pd.to_numeric(df_unido['nf_icms_contabilidade'], errors='coerce').astype('Int64')
df_unido['cfop_icms_contabilidade'] = pd.to_numeric(df_unido['cfop_icms_contabilidade'], errors='coerce').astype('Int64')

### 5.5 Criando as colunas de diferença e status

In [ ]:
df_unido = df_unido.fillna(0)
df_unido['dif_vl_contabil'] = (df_unido['vl_cont_empresa'] - df_unido['vl_cont_icms_contabilidade']).round(2)
df_unido['dif_vl_icms'] = (df_unido['vl_icms_empresa'] - df_unido['vl_icms_contabilidade']).round(2)
df_unido['dif_vl_ipi'] = (df_unido['vl_ipi_empresa'] - df_unido['vl_ipi_contabilidade']).round(2)

cols_dif = ['dif_vl_contabil', 'dif_vl_icms', 'dif_vl_ipi','dif_vl_ipi']
df_unido['status'] = 'Valor Divergente'
df_unido.loc[df_unido[cols_dif].eq(0).all(axis=1), 'status'] = 'OK'
df_unido = df_unido.sort_values(by=['nf', 'cfop'])

### 5.6 Criando a base para exportação para o Excel

In [ ]:
cols_desejadas = ['nf','cfop','vl_cont_empresa','vl_icms_empresa','vl_ipi_empresa', 'vl_cont_icms_contabilidade', 'vl_icms_contabilidade', 'vl_ipi_contabilidade', 'dif_vl_contabil', 'dif_vl_icms', 'dif_vl_ipi','status']
cols_existentes = [col for col in cols_desejadas if col in df_unido.columns]
df_export_excel = df_unido[cols_existentes]

## 6.0 DataFrame resumo

### 6.1 Criação do dataframe df_resumo e fazendo a exportação do arquivo em excel

In [ ]:
df_resumo = pd.DataFrame({
    'Qtde Nfs Empresa': [qtde_nf_empresa],
    'Qtde Nfs Contabilidade': [qtde_nf_icms_contabilidade],
    'Diferença qtde Nfs': [qtde_nf_empresa - qtde_nf_icms_contabilidade],
    
    'Valor Total Contabil Empresa': [valor_contabil_total_empresa],
    'Valor Total Contabil Contabilidade': [valor_contabil_total_icms_contabilidade],
    'Diferença valor total Contabil': [valor_contabil_total_empresa - valor_contabil_total_icms_contabilidade],
    
    'Valor Total ICMS Empresa': [valor_icms_total_empresa],
    'Valor Total ICMS Contabilidade': [valor_icms_total_contabilidade],
    'Diferença Valor total ICMS': [valor_icms_total_empresa - valor_icms_total_contabilidade],
    
    'Valor Total IPI Empresa': [valor_ipi_total_empresa],
    'Valor Total IPI Contabilidade': [valor_ipi_total_contabilidade],
    'Diferença Valor total IPI': [valor_ipi_total_empresa - valor_ipi_total_contabilidade]
})
nome_arquivo = 'apuracao.xlsx'

with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
    df_export_excel.to_excel(writer, sheet_name='apuracao', index=False)
    df_resumo.to_excel(writer, sheet_name='resumo', index=False)


## 7.0 Formatação da Tabela em excel

In [ ]:
# 2. Abre o arquivo salvo com o openpyxl para aplicar as cores de fundo
wb = load_workbook(nome_arquivo)
ws = wb.active

# Define as cores de preenchimento (Verde Claro e Vermelho Claro em formato Hexadecimal)
fill_verde = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid") # Verde claro (suave)
fill_vermelho = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid") # Vermelho claro (suave)

# Descobre qual é o número da coluna 'Classificacao'
coluna_classificacao_idx = None
for col_idx, cell in enumerate(ws[1], 1):
    if cell.value == 'status':
        coluna_classificacao_idx = col_idx
        break

# Se encontrou a coluna, percorre as linhas da tabela aplicando a cor correspondente
if coluna_classificacao_idx:
    for row in range(2, ws.max_row + 1):
        cell = ws.cell(row=row, column=coluna_classificacao_idx)
        if cell.value == 'OK':
            cell.fill = fill_verde
        elif cell.value == 'Valor Divergente':
            cell.fill = fill_vermelho

# Salva o arquivo final com as cores aplicadas
wb.save(nome_arquivo)